# Chapter 6 — Data Normalization & Feature Subset Selection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part1_foundations/ch06_data_normalization_feature_subset_selection/ch06_notebook.ipynb)

Companion notebook for **Chapter 6** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Normalization puts every feature on a comparable scale, so one large-valued column cannot dominate a distance calculation.
- Min-Max squeezes values into [0,1]; Z-score centres them at mean 0 with standard deviation 1.
- Always compute the min, max, mean and standard deviation on the TRAINING data and reuse those same numbers at test time.
- Feature selection drops columns that tell you little about the target; Pearson correlation is the quick screening tool.
- Fewer, better features mean faster training, less overfitting and a model you can actually explain.

## Example 1: Min-Max Normalization

This example walks through Min-Max normalization on a small salesman dataset. We define a simple function that takes an array, finds its minimum and maximum, then scales every value proportionally into the [0, 1] range. Notice how the raw sales values (₹2,300 to ₹73,567) and scores (2 to 8) are completely different scales — after normalization, both live in [0, 1]. The last two rows confirm that the range constraint is satisfied. Remember: you MUST save the min and max from training and apply them at test time too — never recompute from test data.

In [1]:
import numpy as np

# Salesman dataset: Total Sales (Rs) and Score for Sale
sales  = np.array([23500, 23500, 22879,  2300, 34678,
                   15687, 18945,  8750, 37489, 73567,
                   52789,  2900,  6570, 21000], dtype=float)
scores = np.array([8, 6, 2, 4, 5, 8, 8, 2, 4, 2, 4, 3, 3, 2], dtype=float)

def minmax_normalize(arr):
    return (arr - arr.min()) / (arr.max() - arr.min())

sales_norm  = minmax_normalize(sales)
scores_norm = minmax_normalize(scores)

print("=== Min-Max Normalization ===")
print(f"Sales  → min: {sales.min():.0f}, max: {sales.max():.0f}")
print(f"Scores → min: {scores.min():.0f}, max: {scores.max():.0f}")
print()
print(f"{'ID':<6} {'Sales (Rs)':>12} {'Score':>8} {'Norm Sales':>12} {'Norm Score':>12}")
print("-" * 55)
for i, (s, sc, sn, scn) in enumerate(zip(sales, scores, sales_norm, scores_norm), 1):
    print(f"S{i:03d}   {s:>12.0f} {sc:>8.0f} {sn:>12.4f} {scn:>12.4f}")
print()
print(f"Norm Sales range : [{sales_norm.min():.4f}, {sales_norm.max():.4f}]")
print(f"Norm Score range : [{scores_norm.min():.4f}, {scores_norm.max():.4f}]")

=== Min-Max Normalization ===
Sales  → min: 2300, max: 73567
Scores → min: 2, max: 8

ID       Sales (Rs)    Score   Norm Sales   Norm Score
-------------------------------------------------------
S001          23500        8       0.2975       1.0000
S002          23500        6       0.2975       0.6667
S003          22879        2       0.2888       0.0000
S004           2300        4       0.0000       0.3333
S005          34678        5       0.4543       0.5000
S006          15687        8       0.1878       1.0000
S007          18945        8       0.2336       1.0000
S008           8750        2       0.0905       0.0000
S009          37489        4       0.4938       0.3333
S010          73567        2       1.0000       0.0000
S011          52789        4       0.7084       0.3333
S012           2900        3       0.0084       0.1667
S013           6570        3       0.0599       0.1667
S014          21000        2       0.2624       0.0000

Norm Sales range : [0.0000, 1.00

## Example 2: Z-Score Normalization & Distance Comparison

This example shows z-score normalization and — critically — shows WHY normalization matters for distance-based algorithms. Without normalization, we ask: which salesman is closest to a query point of (23000 Rs, 6.5 score)? The Euclidean distance is dominated by the Sales column (values in thousands) making the Score column almost irrelevant. After z-score normalization, both columns are on equal footing. See how S001, S002, and S003 get completely different nearest-neighbour rankings before vs after normalization — S002 becomes the closest, which makes intuitive sense since its sales (23500) and score (6) are both close to the query (23000, 6.5).

In [2]:
import numpy as np

sales  = np.array([23500, 23500, 22879, 2300, 34678,
                   15687, 18945, 8750, 37489, 73567,
                   52789, 2900, 6570, 21000], dtype=float)
scores = np.array([8, 6, 2, 4, 5, 8, 8, 2, 4, 2, 4, 3, 3, 2], dtype=float)

def zscore(arr):
    return (arr - arr.mean()) / arr.std()

sales_z  = zscore(sales)
scores_z = zscore(scores)

print("=== Z-Score Normalization ===")
print(f"Sales  → mean: {sales.mean():.2f}, std: {sales.std():.2f}")
print(f"Scores → mean: {scores.mean():.2f}, std: {scores.std():.2f}")
print()
print(f"{'ID':<6} {'Sales (Rs)':>12} {'Score':>8} {'Z-Sales':>10} {'Z-Score':>10}")
print("-" * 50)
for i, (s, sc, sn, scn) in enumerate(zip(sales, scores, sales_z, scores_z), 1):
    print(f"S{i:03d}   {s:>12.0f} {sc:>8.0f} {sn:>10.2f} {scn:>10.2f}")
print()
print(f"After normalization: mean={sales_z.mean():.4f}, std={sales_z.std():.4f}")

# Compare distances for query point (23000, 6.5)
query_raw = np.array([23000.0, 6.5])
query_z   = np.array([(23000 - sales.mean())/sales.std(),
                      (6.5   - scores.mean())/scores.std()])

d_raw = np.sqrt((sales - query_raw[0])**2 + (scores - query_raw[1])**2)
d_z   = np.sqrt((sales_z - query_z[0])**2 + (scores_z - query_z[1])**2)

print("\n--- Euclidean Distance to query (23000, 6.5) ---")
print("Before normalization (Sales dominates):")
print(f"  S001: {d_raw[0]:.2f}   S002: {d_raw[1]:.2f}   S003: {d_raw[2]:.2f}")
print("After Z-score normalization (both features contribute equally):")
print(f"  S001: {d_z[0]:.4f}   S002: {d_z[1]:.4f}   S003: {d_z[2]:.4f}")
print("\nS002 is now correctly identified as nearest neighbour.")

=== Z-Score Normalization ===
Sales  → mean: 24611.00, std: 19150.39
Scores → mean: 4.36, std: 2.22

ID       Sales (Rs)    Score    Z-Sales    Z-Score
--------------------------------------------------
S001          23500        8      -0.06       1.64
S002          23500        6      -0.06       0.74
S003          22879        2      -0.09      -1.06
S004           2300        4      -1.17      -0.16
S005          34678        5       0.53       0.29
S006          15687        8      -0.47       1.64
S007          18945        8      -0.30       1.64
S008           8750        2      -0.83      -1.06
S009          37489        4       0.67      -0.16
S010          73567        2       2.56      -1.06
S011          52789        4       1.47      -0.16
S012           2900        3      -1.13      -0.61
S013           6570        3      -0.94      -0.61
S014          21000        2      -0.19      -1.06

After normalization: mean=0.0000, std=1.0000

--- Euclidean Distance to query (230

## Example 3: Pearson Correlation & Feature Selection

This example builds a synthetic weather dataset with three features (Temperature, Humidity, Pressure) and a target (Rain). It then computes the Pearson correlation coefficient between each feature and the target to decide which features to keep. The output confirms the intuition: Temperature and Humidity strongly predict Rain (very strong correlation), while Pressure has a weak relationship and could be dropped. The full correlation matrix also reveals that Temperature and Humidity are highly correlated with each other (ρ = -0.925), meaning they carry redundant information — an advanced practitioner might even drop one of them too.

In [3]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 50
temperature = np.random.uniform(20, 35, n)
humidity    = 120 - 1.5 * temperature + np.random.normal(0, 3, n)
pressure    = np.random.uniform(995, 1040, n)
rain        = -2*temperature + 0.8*humidity + np.random.normal(0, 5, n)

df = pd.DataFrame({'Temperature': temperature, 'Humidity': humidity,
                   'Pressure': pressure, 'Rain': rain})

print("=== Pearson Correlation with Target (Rain) ===\n")
corr_with_rain = df.corr()['Rain'].drop('Rain')
for feat, val in corr_with_rain.items():
    strength = ('None' if abs(val) < 0.1 else 'Weak' if abs(val) < 0.3 else
                'Moderate' if abs(val) < 0.5 else 'Strong' if abs(val) < 0.7
                else 'Very Strong')
    direction = 'positive' if val > 0 else 'negative'
    print(f"  rho(Rain, {feat:<12}) = {val:7.4f}  -> {strength} {direction}")

print()
print("=== Full Correlation Matrix ===")
print(df.corr().round(3).to_string())
print()
print("=== Feature Selection Decision (drop threshold: |rho| < 0.15) ===")
for feat, val in corr_with_rain.items():
    action = "KEEP" if abs(val) >= 0.15 else "DROP (weak correlation with target)"
    print(f"  {feat:<14}: rho = {val:+.4f}  -> {action}")

=== Pearson Correlation with Target (Rain) ===

  rho(Rain, Temperature ) = -0.9476  -> Very Strong negative
  rho(Rain, Humidity    ) =  0.9169  -> Very Strong positive
  rho(Rain, Pressure    ) = -0.1530  -> Weak negative

=== Full Correlation Matrix ===
             Temperature  Humidity  Pressure   Rain
Temperature        1.000    -0.925     0.115 -0.948
Humidity          -0.925     1.000    -0.180  0.917
Pressure           0.115    -0.180     1.000 -0.153
Rain              -0.948     0.917    -0.153  1.000

=== Feature Selection Decision (drop threshold: |rho| < 0.15) ===
  Temperature   : rho = -0.9476  -> KEEP
  Humidity      : rho = +0.9169  -> KEEP
  Pressure      : rho = -0.1530  -> KEEP
